COMP64702 - RAG Culinary Assistant
Notebook 05 : Complete Evaluation — All Metrics + Baseline
ALL metrics computed for BOTH systems:
   - Our full RAG system
   - No-context baseline (LLM only, no retrieval)


In [1]:
import os
import sys
import json
import time
import re
import pickle
import numpy as np
import faiss
import torch
from datetime import datetime
from collections import defaultdict
from scipy import stats
from dotenv import load_dotenv
from huggingface_hub import login
 
BASE_DIR = r"D:\text mining\rag project"
os.chdir(BASE_DIR)
sys.path.insert(0, os.path.join(BASE_DIR, "src"))
 
from embedder  import Embedder
from retriever import Retriever
from generator import Generator, build_context
 
# Metric libraries
from rouge_score  import rouge_scorer
from bert_score   import score as bert_score_fn
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
 
import nltk
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet",   quiet=True)
nltk.download("omw-1.4",   quiet=True)
from nltk.translate.bleu_score  import sentence_bleu, SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
 
os.makedirs("outputs/evaluation", exist_ok=True)
 
load_dotenv()
login(token=os.getenv("HF_TOKEN"))
print("Setup complete")
 

Setup complete


In [2]:
# Load models and data
print("Loading models...")
 
embedder_  = Embedder("BAAI/bge-small-en-v1.5")
retriever_ = Retriever("vector_store", embedder=embedder_)
generator_ = Generator("Qwen/Qwen2.5-0.5B-Instruct")
device     = "cuda" if torch.cuda.is_available() else "cpu"
 
# Load train benchmark
with open("data/benchmark/train_set.json", encoding="utf-8") as f:
    train_set = json.load(f)
 
# Load existing RAG outputs (from notebook 04)
with open("outputs/train_outputs.json", encoding="utf-8") as f:
    rag_outputs = json.load(f)
 
# Build source lookup: question -> gold source title
source_lookup = {qa["question"]: qa.get("source_title", "") for qa in train_set}
 
print(f"Train set      : {len(train_set)} QA pairs")
print(f"RAG outputs    : {len(rag_outputs)} predictions")
print("All models loaded\n")
 

Loading models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder loaded: BAAI/bge-small-en-v1.5 (dim=384)
FAISS index loaded — 3,356 vectors
Chunks loaded     — 3,356 chunks
BM25 index loaded


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded
Loading Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generator loaded on cpu
Train set      : 92 QA pairs
RAG outputs    : 92 predictions
All models loaded



In [3]:
# Generate no-context baseline outputs (if not already done)
 
BASELINE_PATH = "outputs/evaluation/baseline_gen_outputs.json"
 
if os.path.exists(BASELINE_PATH):
    print(f"Loading existing baseline outputs from {BASELINE_PATH}")
    with open(BASELINE_PATH, encoding="utf-8") as f:
        baseline_outputs = json.load(f)
    print(f"Loaded {len(baseline_outputs)} baseline predictions")
else:
    print(f"Generating no-context baseline on {len(train_set)} queries...")
    print("(This takes ~5 minutes)\n")
 
    NO_CTX_SYSTEM = (
        "You are a knowledgeable culinary assistant specialising in East Asian cuisine. "
        "Answer the question as best you can from your own knowledge. "
        "Keep your answer between 2 and 5 sentences."
    )
 
    baseline_outputs = []
    for i, qa in enumerate(train_set, 1):
        q = qa["question"]
        messages = [
            {"role": "system", "content": NO_CTX_SYSTEM},
            {"role": "user",   "content": f"Question: {q}\n\nAnswer:"},
        ]
        text   = generator_.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = generator_.tokenizer([text], return_tensors="pt").to(device)
 
        t0 = time.time()
        with torch.no_grad():
            out = generator_.model.generate(
                **inputs, max_new_tokens=200, temperature=0.3,
                do_sample=True, pad_token_id=generator_.tokenizer.eos_token_id,
                repetition_penalty=1.1,
            )
        elapsed = time.time() - t0
 
        generated = out[0][inputs["input_ids"].shape[1]:]
        answer    = generator_.tokenizer.decode(generated, skip_special_tokens=True).strip()
 
        baseline_outputs.append({
            "id":           qa.get("id", f"Q{i}"),
            "question":     q,
            "gold_answer":  qa["answer"],
            "pred_answer":  answer,
            "type":         qa.get("type", "unknown"),
            "difficulty":   qa.get("difficulty", "unknown"),
            "latency_s":    round(elapsed, 3),
            "retrieved":    [],
        })
 
        if i % 10 == 0:
            print(f"  {i}/{len(train_set)} done")
 
    with open(BASELINE_PATH, "w", encoding="utf-8") as f:
        json.dump(baseline_outputs, f, ensure_ascii=False, indent=2)
    print(f"\nBaseline outputs saved to {BASELINE_PATH}")
 

Loading existing baseline outputs from outputs/evaluation/baseline_gen_outputs.json
Loaded 92 baseline predictions


In [4]:
# Measure latency for RAG outputs (if not stored)
 
# Check if latency already recorded
if "latency_s" not in rag_outputs[0]:
    print("Measuring RAG latency on 20 sample queries...")
    latencies = []
    for qa in train_set[:20]:
        t0        = time.time()
        retrieved = retriever_.retrieve(qa["question"], initial_k=20, final_k=5)
        context   = build_context(retrieved, max_words=600)
        _         = generator_.generate(
            query=qa["question"], context=context,
            strategy="structured", max_new_tokens=200, temperature=0.3,
        )
        latencies.append(time.time() - t0)
 
    rag_mean_latency  = float(np.mean(latencies))
    base_mean_latency = float(np.mean([o.get("latency_s", 0) for o in baseline_outputs]))
    print(f"RAG mean latency      : {rag_mean_latency:.2f}s")
    print(f"Baseline mean latency : {base_mean_latency:.2f}s")
else:
    rag_mean_latency  = float(np.mean([o.get("latency_s", 0) for o in rag_outputs]))
    base_mean_latency = float(np.mean([o.get("latency_s", 0) for o in baseline_outputs]))
    latencies         = [o.get("latency_s", 0) for o in rag_outputs]
    print(f"RAG mean latency      : {rag_mean_latency:.2f}s")
    print(f"Baseline mean latency : {base_mean_latency:.2f}s")
 
# Store per-query latencies for significance test
base_latencies = [o.get("latency_s", base_mean_latency) for o in baseline_outputs]
 

Measuring RAG latency on 20 sample queries...
RAG mean latency      : 11.77s
Baseline mean latency : 0.00s


In [5]:
# All metric functions
 
smooth = SmoothingFunction().method1
 
def _normalise(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text)
 
def _normalise_set(text):
    return set(_normalise(text).split())
 
def _tokens(text):
    return _normalise(text).split()
 
# ── 1. BLEU ──────────────────────────────────────────────────────────────────
def compute_bleu(predictions, references):
    """
    BLEU-1, BLEU-2, BLEU-4 using NLTK.
    BLEU measures n-gram precision between prediction and reference.
    Includes brevity penalty for short predictions.
    Higher is better. Range 0-1.
    """
    b1, b2, b4 = [], [], []
    for pred, ref in zip(predictions, references):
        pred_toks = _tokens(pred)
        ref_toks  = [_tokens(ref)]
        if not pred_toks:
            b1.append(0.0); b2.append(0.0); b4.append(0.0)
            continue
        b1.append(sentence_bleu(ref_toks, pred_toks, weights=(1,0,0,0),     smoothing_function=smooth))
        b2.append(sentence_bleu(ref_toks, pred_toks, weights=(0.5,0.5,0,0), smoothing_function=smooth))
        b4.append(sentence_bleu(ref_toks, pred_toks, weights=(0.25,)*4,     smoothing_function=smooth))
    return (
        {"bleu1": b1, "bleu2": b2, "bleu4": b4},
        {"bleu1": float(np.mean(b1)), "bleu2": float(np.mean(b2)), "bleu4": float(np.mean(b4))}
    )
 
# ── 2. ROUGE ─────────────────────────────────────────────────────────────────
def compute_rouge(predictions, references):
    sc = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        r = sc.score(ref, pred)
        r1.append(r["rouge1"].fmeasure)
        r2.append(r["rouge2"].fmeasure)
        rl.append(r["rougeL"].fmeasure)
    return (
        {"rouge1": r1, "rouge2": r2, "rougeL": rl},
        {"rouge1": float(np.mean(r1)), "rouge2": float(np.mean(r2)), "rougeL": float(np.mean(rl))}
    )
 
# ── 3. METEOR ────────────────────────────────────────────────────────────────
def compute_meteor(predictions, references):
    scores = []
    for pred, ref in zip(predictions, references):
        try:
            s = meteor_score([ref.lower().split()], pred.lower().split())
        except Exception:
            s = 0.0
        scores.append(float(s))
    return scores, float(np.mean(scores))
 
# ── 4. BERTScore ─────────────────────────────────────────────────────────────
def compute_bertscore(predictions, references):
    print("  Computing BERTScore...")
    _, _, F1 = bert_score_fn(predictions, references,
                             model_type="distilbert-base-uncased",
                             lang="en", verbose=False)
    scores = F1.numpy().tolist()
    return scores, float(np.mean(scores))
 
# ── 5. Answer F1 ─────────────────────────────────────────────────────────────
def compute_answer_f1(predictions, references):
    def _f1(pred, ref):
        pt = set(pred.lower().split()); rt = set(ref.lower().split())
        c = pt & rt
        if not c: return 0.0
        p = len(c)/len(pt); r = len(c)/len(rt)
        return 2*p*r/(p+r)
    scores = [_f1(p,r) for p,r in zip(predictions, references)]
    return scores, float(np.mean(scores))
 
# ── 6. Exact Match ───────────────────────────────────────────────────────────
def compute_exact_match(predictions, references):
    scores = [1.0 if _normalise(p)==_normalise(r) else 0.0
              for p,r in zip(predictions, references)]
    return scores, float(np.mean(scores))
 
# ── 7. Answer Relevance ──────────────────────────────────────────────────────
def compute_answer_relevance(questions, predictions):
    print("  Computing Answer Relevance...")
    q_embs = embedder_.encode_documents(questions,   show_progress=False)
    a_embs = embedder_.encode_documents(predictions, show_progress=False)
    scores = [float(sklearn_cosine(q.reshape(1,-1), a.reshape(1,-1))[0][0])
              for q,a in zip(q_embs, a_embs)]
    return scores, float(np.mean(scores))
 
# ── 8. Answer Correctness ────────────────────────────────────────────────────
def compute_answer_correctness(predictions, references, questions):
    """
    Answer Correctness = harmonic mean of:
      - BERTScore F1 (semantic similarity to gold answer)
      - Answer Relevance (semantic similarity to question)
    Combines both into one holistic quality score.
    """
    print("  Computing Answer Correctness...")
    _, bs_mean     = compute_bertscore(predictions, references)
    ar_scores, _   = compute_answer_relevance(questions, predictions)
    bs_scores_list, _ = compute_bertscore(predictions, references)
 
    # Per-query harmonic mean
    correctness = []
    for bs, ar in zip(bs_scores_list, ar_scores):
        if bs + ar > 0:
            correctness.append(2 * bs * ar / (bs + ar))
        else:
            correctness.append(0.0)
    return correctness, float(np.mean(correctness))
 

# ── 10. Context Precision ────────────────────────────────────────────────────
def compute_context_precision(outputs, source_lookup):
    scores = []
    for o in outputs:
        gold      = source_lookup.get(o["question"],"").lower()
        retrieved = o.get("retrieved",[])
        if not gold or not retrieved: scores.append(0.0); continue
        rel = sum(1 for c in retrieved
                  if gold in c.get("doc_title","").lower()
                  or c.get("doc_title","").lower() in gold)
        scores.append(rel / len(retrieved))
    return scores, float(np.mean(scores))
 
# ── 11. Retrieval metrics: MRR@K, Recall@K, NDCG@K ──────────────────────────
def compute_retrieval_metrics(outputs, source_lookup, k_values=(1,3,5,10)):
    """
    MRR@K     : 1/rank of first relevant chunk in top-K
    Recall@K  : fraction of queries where gold doc appears in top-K
    Precision@K: fraction of top-K chunks that are relevant
    NDCG@K    : normalised discounted cumulative gain
    """
    raw = {k: defaultdict(list) for k in k_values}
 
    for o in outputs:
        gold      = source_lookup.get(o["question"],"").lower()
        retrieved = o.get("retrieved",[])
        if not gold or not retrieved: continue
 
        relevance = [
            1 if (gold in c.get("doc_title","").lower()
                  or c.get("doc_title","").lower() in gold) else 0
            for c in retrieved
        ]
 
        for k in k_values:
            top_k = relevance[:k]
 
            # MRR
            mrr = next((1/r for r,rel in enumerate(top_k,1) if rel), 0.0)
            raw[k]["mrr"].append(mrr)
 
            # Recall
            raw[k]["recall"].append(1.0 if any(top_k) else 0.0)
 
            # Precision
            raw[k]["precision"].append(sum(top_k)/k)
 
            # NDCG
            dcg  = sum(r/np.log2(i+1) for i,r in enumerate(top_k,1))
            idcg = sum(r/np.log2(i+1) for i,r in enumerate(sorted(top_k,reverse=True),1))
            raw[k]["ndcg"].append(dcg/idcg if idcg>0 else 0.0)
 
    means = {k: {m: float(np.mean(v)) if v else 0.0 for m,v in raw[k].items()}
             for k in k_values}
    return raw, means
 
# ── 12. Statistical significance ─────────────────────────────────────────────
def sig_test(system_scores, baseline_scores, metric_name=""):
    n = min(len(system_scores), len(baseline_scores))
    if n < 5:
        return 0.0, 1.0, False, "—"
    # Check if both arrays are constant — t-test is undefined
    if np.std(system_scores[:n]) == 0 and np.std(baseline_scores[:n]) == 0:
        return 0.0, 1.0, False, "— (both zero)"
    try:
        t, p = stats.ttest_rel(system_scores[:n], baseline_scores[:n])
        if np.isnan(p):
            return 0.0, 1.0, False, "— (undefined)"
        sig = bool(p < 0.05)
        if p < 0.001:   sym = "*** p<0.001"
        elif p < 0.01:  sym = "**  p<0.01"
        elif p < 0.05:  sym = "*   p<0.05"
        else:           sym = f"    p={p:.3f}"
        return float(t), float(p), sig, sym
    except Exception:
        return 0.0, 1.0, False, "—"
 

In [7]:
import pickle

# Load full chunks once at the top of the cell
with open("vector_store/chunks.pkl", "rb") as f:
    full_chunks = pickle.load(f)

# Build a lookup: chunk_id -> full text
chunk_lookup = {c["chunk_id"]: c["text"] for c in full_chunks}

matched = 0
unmatched = 0
for o in rag_outputs:
    for chunk in o.get("retrieved", []):
        cid = chunk.get("chunk_id", "")
        if cid in chunk_lookup:
            matched += 1
        else:
            unmatched += 1

print(f"Chunk ID matches   : {matched}")
print(f"Chunk ID mismatches: {unmatched}")
print(f"\nSample chunk_id from outputs: {rag_outputs[0]['retrieved'][0].get('chunk_id','MISSING')}")
print(f"Sample chunk_id from pkl   : {list(chunk_lookup.keys())[0]}")

Chunk ID matches   : 460
Chunk ID mismatches: 0

Sample chunk_id from outputs: wikibooks_Cookbook:Sushi_1
Sample chunk_id from pkl   : wikipedia_Amazake_0


In [8]:

STOPWORDS = {
    'i','me','my','we','our','you','your','he','him','his','she','her',
    'it','its','they','them','their','what','which','who','this','that',
    'these','those','am','is','are','was','were','be','been','being',
    'have','has','had','do','does','did','a','an','the','and','but',
    'if','or','as','of','at','by','for','with','into','through','during',
    'before','after','to','from','up','in','out','on','over','then',
    'when','where','how','all','both','each','more','other','some','such',
    'no','not','only','same','so','than','too','very','can','will',
    'just','should','now','about','also',
}

def content_words(text):
    tokens = re.sub(r'[^\w\s]', '', text.lower()).split()
    return set(t for t in tokens if t not in STOPWORDS and len(t) > 2)

def compute_faithfulness(outputs, chunk_lookup, threshold=0.40):
    """
    Correct faithfulness using:
    1. Full chunk text from chunks.pkl (not truncated 200-char version)
    2. Stopwords removed (content words only)
    3. threshold=0.40 — at least 40% of content words in context
    """
    scores = []

    for o in outputs:
        pred_content    = content_words(o.get("pred_answer", ""))
        context_content = set()

        for chunk in o.get("retrieved", []):
            # Try to get full text from lookup first
            cid       = chunk.get("chunk_id", "")
            full_text = chunk_lookup.get(cid, chunk.get("text", ""))
            context_content.update(content_words(full_text))

        if not pred_content:
            scores.append(0.0)
            continue
        if not context_content:
            scores.append(0.0)
            continue

        overlap = pred_content & context_content
        scores.append(len(overlap) / len(pred_content))

    mean_faith  = float(np.mean(scores))
    hall_rate   = float(sum(1 for s in scores if s < threshold) / max(len(scores), 1))
    hall_scores = [1.0 - s for s in scores]

    return scores, mean_faith, hall_rate, hall_scores

In [9]:
# Compute all metrics for RAG system
 
print("="*60)
print("  COMPUTING METRICS — RAG SYSTEM")
print("="*60)
 
rag_preds  = [o["pred_answer"] for o in rag_outputs]
rag_refs   = [o["gold_answer"] for o in rag_outputs]
rag_qs     = [o["question"]    for o in rag_outputs]
 
print("\nGeneration metrics...")
rag_bleu_raw,   rag_bleu   = compute_bleu(rag_preds, rag_refs)
rag_rouge_raw,  rag_rouge  = compute_rouge(rag_preds, rag_refs)
rag_meteor,     rag_met    = compute_meteor(rag_preds, rag_refs)
print("  BLEU and ROUGE done")
rag_bert,       rag_bs     = compute_bertscore(rag_preds, rag_refs)
rag_af1,        rag_af1m   = compute_answer_f1(rag_preds, rag_refs)
rag_em,         rag_emm    = compute_exact_match(rag_preds, rag_refs)
rag_ar,         rag_arm    = compute_answer_relevance(rag_qs, rag_preds)
rag_ac,         rag_acm    = compute_answer_correctness(rag_preds, rag_refs, rag_qs)
 
print("\nRAG-specific metrics...")
rag_faith, rag_faith_mean, rag_hall_rate, rag_hall_scores = compute_faithfulness(rag_outputs, chunk_lookup, threshold=0.40)

print(f"Faithfulness mean  : {rag_faith_mean:.4f}")
print(f"Hallucination rate : {rag_hall_rate*100:.1f}%")
rag_cp, rag_cpm = compute_context_precision(rag_outputs, source_lookup)
 
print("\nRetrieval metrics...")
rag_ret_raw, rag_ret = compute_retrieval_metrics(rag_outputs, source_lookup, k_values=(1,3,5,10))
 
print("\nAll RAG metrics computed")
 

  COMPUTING METRICS — RAG SYSTEM

Generation metrics...
  BLEU and ROUGE done
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing Answer Relevance...
  Computing Answer Correctness...
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing Answer Relevance...
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



RAG-specific metrics...
Faithfulness mean  : 0.6547
Hallucination rate : 10.9%

Retrieval metrics...

All RAG metrics computed


In [10]:
# Compute all metrics for no-context baseline
 
print("\n" + "="*60)
print("  COMPUTING METRICS — NO-CONTEXT BASELINE")
print("="*60)
 
base_preds = [o["pred_answer"] for o in baseline_outputs]
base_refs  = [o["gold_answer"] for o in baseline_outputs]
base_qs    = [o["question"]    for o in baseline_outputs]
 
print("\nGeneration metrics...")
base_bleu_raw, base_bleu  = compute_bleu(base_preds, base_refs)
base_rouge_raw, base_rouge = compute_rouge(base_preds, base_refs)
base_meteor, base_met     = compute_meteor(base_preds, base_refs)
print("  BLEU and ROUGE done")
base_bert, base_bs        = compute_bertscore(base_preds, base_refs)
base_af1, base_af1m       = compute_answer_f1(base_preds, base_refs)
base_em, base_emm         = compute_exact_match(base_preds, base_refs)
base_ar, base_arm         = compute_answer_relevance(base_qs, base_preds)
base_ac, base_acm         = compute_answer_correctness(base_preds, base_refs, base_qs)
 
print("\nBaseline-specific metrics...")
# Baseline has no retrieval so faithfulness and context metrics are 0
base_faith       = [0.0] * len(baseline_outputs)
base_faith_mean  = 0.0
base_hall_rate   = 1.0   # 100% hallucination rate — no context at all
base_cp          = [0.0] * len(baseline_outputs)
base_cpm         = 0.0
 
# Baseline retrieval metrics — all 0 since no retrieval
base_ret = {k: {"mrr": 0.0, "recall": 0.0, "precision": 0.0, "ndcg": 0.0}
            for k in (1,3,5,10)}
base_ret_raw = {k: {"mrr": [0.0]*len(baseline_outputs),
                    "recall": [0.0]*len(baseline_outputs)}
                for k in (1,3,5,10)}
 
print("\nAll baseline metrics computed")
 


  COMPUTING METRICS — NO-CONTEXT BASELINE

Generation metrics...
  BLEU and ROUGE done
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing Answer Relevance...
  Computing Answer Correctness...
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing Answer Relevance...
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Baseline-specific metrics...

All baseline metrics computed


In [11]:
# Run significance tests for ALL metrics
 
print("\nRunning significance tests...")
 
def sig(rag_s, base_s, name=""):
    return sig_test(rag_s, base_s, name)
 
sig_results = {
    # Generation
    "bleu1":            sig(rag_bleu_raw["bleu1"],   base_bleu_raw["bleu1"]),
    "bleu2":            sig(rag_bleu_raw["bleu2"],   base_bleu_raw["bleu2"]),
    "bleu4":            sig(rag_bleu_raw["bleu4"],   base_bleu_raw["bleu4"]),
    "rouge1":           sig(rag_rouge_raw["rouge1"], base_rouge_raw["rouge1"]),
    "rouge2":           sig(rag_rouge_raw["rouge2"], base_rouge_raw["rouge2"]),
    "rougeL":           sig(rag_rouge_raw["rougeL"], base_rouge_raw["rougeL"]),
    "meteor":           sig(rag_meteor,              base_meteor),
    "bertscore":        sig(rag_bert,                base_bert),
    "answer_f1":        sig(rag_af1,                 base_af1),
    "exact_match":      sig(rag_em,                  base_em),
    "answer_relevance": sig(rag_ar,                  base_ar),
    "answer_correctness":sig(rag_ac,                 base_ac),
    # RAG-specific
    "faithfulness":     sig(rag_faith,               base_faith),
    "hallucination":    sig(base_faith,              rag_faith),  # reversed: lower is better
    # Latency — lower is better for RAG (reversed)
    "latency":          sig_test(base_latencies[:len(latencies)],
                                 latencies[:len(base_latencies)], "latency"),
    # Retrieval — baseline gets 0s for all
    "recall@5":         sig(rag_ret_raw[5]["recall"], base_ret_raw[5]["recall"]),
    "recall@10":        sig(rag_ret_raw[10]["recall"] if 10 in rag_ret_raw else
                           rag_ret_raw[5]["recall"],
                           base_ret_raw[5]["recall"]),
    "mrr@5":            sig(rag_ret_raw[5]["mrr"],    base_ret_raw[5]["mrr"]),
}
 
print("All significance tests done")
 


Running significance tests...
All significance tests done


In [12]:
# Print complete comparison table
 
print("\n" + "="*85)
print("  COMPLETE EVALUATION RESULTS — RAG SYSTEM vs NO-CONTEXT BASELINE")
print("="*85)
 
def fmt(val):
    if isinstance(val, float): return f"{val:.4f}"
    return str(val)
 
# Define all rows: (display_name, metric_key, rag_val, base_val, higher_is_better)
rows = [
    # ── BLEU ──────────────────────────────────────────────────────────────────
    ("─── N-gram metrics ───────────────────────────────────",  "", None, None, True),
    ("BLEU-1",             "bleu1",    rag_bleu["bleu1"],    base_bleu["bleu1"],    True),
    ("BLEU-2",             "bleu2",    rag_bleu["bleu2"],    base_bleu["bleu2"],    True),
    ("BLEU-4",             "bleu4",    rag_bleu["bleu4"],    base_bleu["bleu4"],    True),
 
    # ── ROUGE ─────────────────────────────────────────────────────────────────
    ("─── ROUGE metrics ────────────────────────────────────",  "", None, None, True),
    ("ROUGE-1",            "rouge1",   rag_rouge["rouge1"],  base_rouge["rouge1"],  True),
    ("ROUGE-2",            "rouge2",   rag_rouge["rouge2"],  base_rouge["rouge2"],  True),
    ("ROUGE-L",            "rougeL",   rag_rouge["rougeL"],  base_rouge["rougeL"],  True),
 
    # ── Semantic ──────────────────────────────────────────────────────────────
    ("─── Semantic similarity ──────────────────────────────", "", None, None, True),
    ("BERTScore F1",       "bertscore",rag_bs,               base_bs,               True),
    ("METEOR",             "meteor",   rag_met,              base_met,              True),
 
    # ── QA metrics ────────────────────────────────────────────────────────────
    ("─── QA metrics ───────────────────────────────────────", "", None, None, True),
    ("Answer F1",          "answer_f1",rag_af1m,             base_af1m,             True),
    ("Exact Match",        "exact_match",rag_emm,            base_emm,              True),
    ("Answer Relevance",   "answer_relevance", rag_arm,      base_arm,              True),
    ("Answer Correctness", "answer_correctness", rag_acm,    base_acm,              True),
 
    # ── RAG-specific ──────────────────────────────────────────────────────────
    ("─── RAG-specific metrics ─────────────────────────────", "", None, None, True),
    ("Faithfulness",       "faithfulness", rag_faith_mean,   base_faith_mean,       True),
    ("Hallucination Rate", "hallucination",rag_hall_rate,    base_hall_rate,        False),
    ("Context Precision",  "context_precision", rag_cpm,     base_cpm,              True),
 
    # ── Retrieval ─────────────────────────────────────────────────────────────
    ("─── Retrieval metrics ────────────────────────────────", "", None, None, True),
    ("MRR@1",              "mrr@1",    rag_ret[1]["mrr"],    0.0,                   True),
    ("MRR@3",              "mrr@3",    rag_ret[3]["mrr"],    0.0,                   True),
    ("MRR@5",              "mrr@5",    rag_ret[5]["mrr"],    0.0,                   True),
    ("Recall@5",           "recall@5", rag_ret[5]["recall"], 0.0,                   True),
    ("Recall@10",          "recall@10",rag_ret[10]["recall"] if 10 in rag_ret else rag_ret[5]["recall"], 0.0, True),
    ("NDCG@5",             "ndcg@5",   rag_ret[5]["ndcg"],   0.0,                   True),
    ("NDCG@10",            "ndcg@10",  rag_ret[10]["ndcg"] if 10 in rag_ret else rag_ret[5]["ndcg"], 0.0, True),
 
    # ── Latency ───────────────────────────────────────────────────────────────
    ("─── Efficiency ───────────────────────────────────────", "", None, None, True),
    ("Mean latency (s)",   "latency",  rag_mean_latency,     base_mean_latency,     False),
]
 
print(f"\n  {'Metric':<28} {'RAG System':>12} {'Baseline':>12} {'Delta':>10} {'Sig.':>14}")
print("  " + "─"*80)
 
for row in rows:
    name, key, rag_val, base_val, higher_better = row
 
    # Section divider
    if rag_val is None:
        print(f"\n  {name}")
        continue
 
    delta = rag_val - base_val
    sign  = "+" if delta >= 0 else ""
 
    # Determine winner marker
    if higher_better:
        win = "RAG" if rag_val > base_val else ("BASE" if base_val > rag_val else "TIED")
    else:
        win = "RAG" if rag_val < base_val else ("BASE" if base_val < rag_val else "TIED")
 
    # Significance symbol
    sig_sym = "—"
    if key in sig_results:
        sig_sym = sig_results[key][3]
    elif key in ("mrr@1","mrr@3","ndcg@10","recall@10"):
        sig_sym = "(no baseline)"
 
    win_str = f"[{win}]" if win != "TIED" else ""
    print(f"  {name:<28} {fmt(rag_val):>12} {fmt(base_val):>12} "
          f"{sign}{fmt(abs(delta)):>9}  {sig_sym:<16} {win_str}")
 
print("\n  *** p<0.001  ** p<0.01  * p<0.05")
 


  COMPLETE EVALUATION RESULTS — RAG SYSTEM vs NO-CONTEXT BASELINE

  Metric                         RAG System     Baseline      Delta           Sig.
  ────────────────────────────────────────────────────────────────────────────────

  ─── N-gram metrics ───────────────────────────────────
  BLEU-1                             0.2847       0.1325 +   0.1522  *** p<0.001      [RAG]
  BLEU-2                             0.2092       0.0629 +   0.1463  *** p<0.001      [RAG]
  BLEU-4                             0.1393       0.0245 +   0.1148  *** p<0.001      [RAG]

  ─── ROUGE metrics ────────────────────────────────────
  ROUGE-1                            0.3725       0.2145 +   0.1580  *** p<0.001      [RAG]
  ROUGE-2                            0.2095       0.0556 +   0.1539  *** p<0.001      [RAG]
  ROUGE-L                            0.3280       0.1543 +   0.1737  *** p<0.001      [RAG]

  ─── Semantic similarity ──────────────────────────────
  BERTScore F1                       0.8

In [13]:
# Per-type and per-difficulty breakdown (RAG only)
 
print("\n" + "="*65)
print("  RAG SYSTEM — RESULTS BY QUESTION TYPE")
print("="*65)
 
type_groups = defaultdict(lambda: defaultdict(list))
diff_groups = defaultdict(lambda: defaultdict(list))
 
for i, o in enumerate(rag_outputs):
    qt   = o.get("type",       "unknown")
    qd   = o.get("difficulty", "unknown")
    if i < len(rag_rouge_raw["rougeL"]):
        type_groups[qt]["rougeL"].append(rag_rouge_raw["rougeL"][i])
        type_groups[qt]["bertscore"].append(rag_bert[i])
        type_groups[qt]["bleu1"].append(rag_bleu_raw["bleu1"][i])
        type_groups[qt]["meteor"].append(rag_meteor[i])
        diff_groups[qd]["rougeL"].append(rag_rouge_raw["rougeL"][i])
        diff_groups[qd]["bertscore"].append(rag_bert[i])
 
print(f"\n  {'Type':<16} {'N':>4} {'ROUGE-L':>9} {'BERTScore':>10} {'BLEU-1':>8} {'METEOR':>8}")
print("  " + "─"*57)
for qt, vals in sorted(type_groups.items()):
    n = len(vals["rougeL"])
    print(f"  {qt:<16} {n:>4} "
          f"{np.mean(vals['rougeL']):>9.4f} "
          f"{np.mean(vals['bertscore']):>10.4f} "
          f"{np.mean(vals['bleu1']):>8.4f} "
          f"{np.mean(vals['meteor']):>8.4f}")
 
print(f"\n  {'Difficulty':<12} {'N':>4} {'ROUGE-L':>9} {'BERTScore':>10}")
print("  " + "─"*38)
for qd, vals in sorted(diff_groups.items()):
    print(f"  {qd:<12} {len(vals['rougeL']):>4} "
          f"{np.mean(vals['rougeL']):>9.4f} "
          f"{np.mean(vals['bertscore']):>10.4f}")
 


  RAG SYSTEM — RESULTS BY QUESTION TYPE

  Type                N   ROUGE-L  BERTScore   BLEU-1   METEOR
  ─────────────────────────────────────────────────────────
  comparative         2    0.2232     0.8384   0.2220   0.2104
  cultural           27    0.3415     0.8267   0.2846   0.3119
  factual            21    0.3454     0.8374   0.3062   0.3345
  ingredient         16    0.3332     0.8464   0.2817   0.3400
  procedural         26    0.3047     0.8289   0.2741   0.3148

  Difficulty      N   ROUGE-L  BERTScore
  ──────────────────────────────────────
  easy           25    0.3112     0.8276
  hard           30    0.3576     0.8371
  medium         37    0.3153     0.8344


In [14]:
# Summary scorecard
 
sig_wins = sum(1 for k,v in sig_results.items() if v[2])
 
print("\n" + "="*65)
print("  SUMMARY SCORECARD")
print("="*65)
print(f"""
  Total metrics evaluated  : {len([r for r in rows if r[2] is not None])}
  Metrics where RAG wins   : {sum(1 for r in rows if r[2] is not None and
                                   ((r[4] and r[2] > r[3]) or (not r[4] and r[2] < r[3])))}
  Significant improvements : {sig_wins} (p < 0.05)
 
  Key highlights:
    BLEU-1          : {rag_bleu['bleu1']:.4f}  vs baseline {base_bleu['bleu1']:.4f}  ({sig_results['bleu1'][3].strip()})
    ROUGE-L         : {rag_rouge['rougeL']:.4f}  vs baseline {base_rouge['rougeL']:.4f}  ({sig_results['rougeL'][3].strip()})
    BERTScore F1    : {rag_bs:.4f}  vs baseline {base_bs:.4f}  ({sig_results['bertscore'][3].strip()})
    METEOR          : {rag_met:.4f}  vs baseline {base_met:.4f}  ({sig_results['meteor'][3].strip()})
    Answer Correct. : {rag_acm:.4f}  vs baseline {base_acm:.4f}  ({sig_results['answer_correctness'][3].strip()})
    Faithfulness    : {rag_faith_mean:.4f}  vs baseline {base_faith_mean:.4f}
    Hallucination % : {rag_hall_rate*100:.1f}%   vs baseline {base_hall_rate*100:.1f}%
    Recall@5        : {rag_ret[5]['recall']:.4f}  vs baseline 0.0000
    MRR@5           : {rag_ret[5]['mrr']:.4f}  vs baseline 0.0000
    Mean Latency    : {rag_mean_latency:.2f}s    vs baseline {base_mean_latency:.2f}s
""")
 


  SUMMARY SCORECARD

  Total metrics evaluated  : 23
  Metrics where RAG wins   : 20
  Significant improvements : 16 (p < 0.05)

  Key highlights:
    BLEU-1          : 0.2847  vs baseline 0.1325  (*** p<0.001)
    ROUGE-L         : 0.3280  vs baseline 0.1543  (*** p<0.001)
    BERTScore F1    : 0.8334  vs baseline 0.7815  (*** p<0.001)
    METEOR          : 0.3206  vs baseline 0.2184  (*** p<0.001)
    Answer Correct. : 0.8398  vs baseline 0.8141  (*** p<0.001)
    Faithfulness    : 0.6547  vs baseline 0.0000
    Hallucination % : 10.9%   vs baseline 100.0%
    Recall@5        : 0.9783  vs baseline 0.0000
    MRR@5           : 0.9080  vs baseline 0.0000
    Mean Latency    : 11.77s    vs baseline 0.00s



In [15]:
# Save complete evaluation report
 
import numpy as np
 
def to_python(obj):
    """Convert numpy types for JSON serialisation."""
    if isinstance(obj, dict):    return {k: to_python(v) for k,v in obj.items()}
    if isinstance(obj, list):    return [to_python(v) for v in obj]
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.bool_,)):    return bool(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    return obj
 
report = {
    "created_at":   datetime.now().isoformat(),
    "num_queries":  len(rag_outputs),
    "rag_system": {
        "bleu1":             rag_bleu["bleu1"],
        "bleu2":             rag_bleu["bleu2"],
        "bleu4":             rag_bleu["bleu4"],
        "rouge1":            rag_rouge["rouge1"],
        "rouge2":            rag_rouge["rouge2"],
        "rougeL":            rag_rouge["rougeL"],
        "meteor":            rag_met,
        "bertscore":         rag_bs,
        "answer_f1":         rag_af1m,
        "exact_match":       rag_emm,
        "answer_relevance":  rag_arm,
        "answer_correctness":rag_acm,
        "faithfulness":      rag_faith_mean,
        "hallucination_rate":rag_hall_rate,
        "context_precision": rag_cpm,
        "mrr_at_1":          rag_ret[1]["mrr"],
        "mrr_at_3":          rag_ret[3]["mrr"],
        "mrr_at_5":          rag_ret[5]["mrr"],
        "recall_at_5":       rag_ret[5]["recall"],
        "recall_at_10":      rag_ret[10]["recall"] if 10 in rag_ret else rag_ret[5]["recall"],
        "ndcg_at_5":         rag_ret[5]["ndcg"],
        "mean_latency_s":    rag_mean_latency,
    },
    "baseline": {
        "bleu1":             base_bleu["bleu1"],
        "bleu2":             base_bleu["bleu2"],
        "bleu4":             base_bleu["bleu4"],
        "rouge1":            base_rouge["rouge1"],
        "rouge2":            base_rouge["rouge2"],
        "rougeL":            base_rouge["rougeL"],
        "meteor":            base_met,
        "bertscore":         base_bs,
        "answer_f1":         base_af1m,
        "exact_match":       base_emm,
        "answer_relevance":  base_arm,
        "answer_correctness":base_acm,
        "faithfulness":      0.0,
        "hallucination_rate":1.0,
        "context_precision": 0.0,
        "mrr_at_5":          0.0,
        "recall_at_5":       0.0,
        "recall_at_10":      0.0,
        "ndcg_at_5":         0.0,
        "mean_latency_s":    base_mean_latency,
    },
    "significance_tests": {
        k: {
            "t_stat":      v[0],
            "p_value":     v[1],
            "significant": v[2],
            "label":       v[3],
        }
        for k, v in sig_results.items()
    },
}
 
report = to_python(report)
out_path = "outputs/evaluation/evaluation_report_complete.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
 
print(f"Complete evaluation report saved to {out_path}")
print("\nEvaluation complete.")

Complete evaluation report saved to outputs/evaluation/evaluation_report_complete.json

Evaluation complete.
